Load and explore the dataset

In [2]:
import pandas as pd 
import re

In [3]:
#### load data and remove duplicate
meta_qa = pd.read_parquet("parquet_data/qa_nl.parquet")
print(f'duplicated:', meta_qa.question.duplicated().sum())
meta_qa = meta_qa.drop_duplicates(subset='question', keep='first').reset_index(drop=True)
meta_qa.shape

duplicated: 172


(1263, 6)

In [4]:
meta_qa.keys()

Index(['id', 'question', 'answer', 'regions', 'topics', 'article_ids'], dtype='object')

In [5]:
num_words = meta_qa.answer.str.split().str.len().tolist()

In [6]:
pd.Series(num_words).describe()

count    1263.000000
mean      245.606492
std       116.353604
min        12.000000
25%       160.500000
50%       227.000000
75%       314.500000
max       720.000000
dtype: float64

In [7]:
### the questions contain 23 acronyms
acronym_pattern = re.compile(
    r"\b(?:"
        r"(?:[A-ZÁÀÉÈËÏÖÜÓÚ]{1,3}\.){2,}"   # A.O.W. B.T.W. V.N.
        r"|"
        r"[A-ZÁÀÉÈËÏÖÜÓÚ]{2,}"              # NAVO RIVM AOW COVID-19
        r")\b"
)
acronyms_query = (
    meta_qa["question"]
    .apply(lambda text: acronym_pattern.findall(text))
)

acronyms_query = sorted(set(acr for lst in acronyms_query for acr in lst))

print(acronyms_query)

['APA', 'CSR', 'DAVO', 'EPC', 'EU', 'GIB', 'GPMI', 'IT', 'IVT', 'MI', 'OCMW', 'RIZIV', 'RMI', 'RVA', 'THAB']


In [8]:
meta_qa["num_articles"] = meta_qa.article_ids.apply(lambda x: len(x.split(" ")))

In [9]:
meta_qa.num_articles.describe()

count    1263.000000
mean        2.159145
std         1.836635
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        31.000000
Name: num_articles, dtype: float64

In [10]:
explicit_queries = meta_qa[meta_qa.num_articles == 1]
implicit_queries = meta_qa[meta_qa.num_articles > 1]

print(f'the number of explicit_queries:', len(explicit_queries))
print(f'the number of implicit_queries:', len(implicit_queries))

the number of explicit_queries: 576
the number of implicit_queries: 687


In [11]:
meta_qa.regions.value_counts()

regions
['Waals Gewest', 'Brussels Hoofdstedelijk Gewest', 'Vlaams Gewest']    794
['Waals Gewest']                                                       269
['Brussels Hoofdstedelijk Gewest']                                     151
['Waals Gewest', 'Brussels Hoofdstedelijk Gewest']                      49
Name: count, dtype: int64

In [12]:
meta_qa.topics.value_counts()

topics
Familie;Situatie van koppels;Huwelijk                                                                                         36
Vreemdelingen;Vreemdelingen en familie                                                                                        25
Familie;Situatie van koppels;Wettelijk samenwonen                                                                             25
Familie;Onderhoudsverplichtingen;Onderhoudsverplichtingen (voor kinderen)                                                     23
Huisvesting;Ongezondheid in Wallonië                                                                                          18
                                                                                                                              ..
Privéleven;Rechten van de patiënt;Einde van het leven en euthanasie                                                            1
Huisvesting;Huur in Wallonië;Woonruimte delen (Wallonië);Reparaties;onderhoud en werkzaamh

SimpleRAG

In [ ]:
sim_cor = pd.read_parquet("output/judgeRAG/correctness_results.parquet")
print(f'duplicated:', sim_cor.input.duplicated().sum())
sim_cor = sim_cor.drop_duplicates(subset='input', keep='first').reset_index(drop=True)
print(sim_cor.shape)
sim_cor = sim_cor.set_index('input').reindex(meta_qa['question']).reset_index()
sim_cor

In [ ]:
sim_faith = pd.read_parquet("output/judgeRAG/faithfulness_results.parquet")
print(f'duplicated:', sim_faith.question.duplicated().sum())
sim_faith = sim_faith.drop_duplicates(subset='question', keep='first').reset_index(drop=True)
print(sim_faith.shape)
sim_faith = sim_faith.set_index('question').reindex(meta_qa['question']).reset_index()
sim_faith

In [ ]:
outp = pd.read_parquet("output/judgeRAG/rag_outputs.parquet")
print(f'duplicated:', outp['query'].duplicated().sum())
outp = outp.drop_duplicates(subset='query', keep='first').reset_index(drop=True)
print(outp.shape)
outp = outp.set_index('query').reindex(meta_qa['question']).reset_index()
outp

In [ ]:
sim_cor['faithfulness_score'] = sim_faith['faithfulness_score']
sim_cor['latency_sec'] = outp['latency_sec']
sim_cor['retrieved_ids'] = outp['retrieved_ids']
sim_cor['gold_ids'] = meta_qa['article_ids']
sim_cor['regions'] = meta_qa['regions']
sim_cor['topics'] = meta_qa['topics']
col = meta_qa.id
sim_cor.insert(0, 'id', col)

In [ ]:
sim_cor.to_excel("clean_data/judgeRAG_all_output.xlsx",index=False)